In [1]:
import os
import sys
import argparse

import seaborn as sns
import matplotlib.pyplot as plt

from data import *
from utils import *
import pandas as pd
from scipy import stats
from datetime import datetime

sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Disentanglement'))
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Accuracy'))
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Attack'))

from DCI import compute_dci, report_dci
from IRS import compute_irs_from_multihot_labels, compute_irs
from MIG import compute_mig
from BA import oh_binary_accuracy
from WB import *

## Clean Performance
### Load Models

In [2]:
parser = argparse.ArgumentParser(description='Gen-RKM Model')

parser.add_argument('--N', type=int, default=10_000, help='Total # of samples')                     # Maybe 10_000
parser.add_argument('--mb_size', type=int, default=128, help='Mini-batch size. See utils.py')
parser.add_argument('--h_dim', type=int, default=24, help='Dim of latent vector')                   # Maybe 16, 32
parser.add_argument('--capacity', type=int, default=32, help='Capacity of network. See utils.py')   # Maybe 16, 48 or 64 if too blurry
parser.add_argument('--x_fdim', type=int, default=48, help='Input x_fdim. See utils.py')            # Maybe 96 or 128
parser.add_argument('--y_fdim', type=int, default=16, help='Input y_fdim. See utils.py')            # Maybe 10, 16, 20, 32
parser.add_argument('--c_accu', type=float, default=25, help='Input weight on recons_error')        # Maybe 10, 25, 50, 100

# Training Settings =============================
parser.add_argument('--lr', type=float, default=1e-4, help='Input learning rate for optimizer')     # Maybe 3e-4, 1e-4, 5e-5
parser.add_argument('--max_epochs', type=int, default=0, help='Input max_epoch for cut-off') # 500
parser.add_argument('--device', type=str, default='cpu', help='Device type: cuda or cpu')
parser.add_argument('--workers', type=int, default=0, help='# of workers for dataloader')
parser.add_argument('--shuffle', type=bool, default=True, help='shuffle dataset: true or false')

opt, _ = parser.parse_known_args()

In [3]:
_, xtest, ipVec_dim, nChannels = get_mnist_dataloader(args=opt)

In [4]:
model = '1V'            # 1V or 2V
i_val = 10

In [5]:
def kPCA(X, Y, Z=None, eps=1e-6):
    a = X @ X.T + Y @ Y.T
    B = a.size(0)

    oneN = torch.ones(B, B, device=a.device, dtype=a.dtype) / B
    a = a - oneN @ a - a @ oneN + oneN @ a @ oneN

    a = 0.5 * (a + a.T)  # numerical symmetry
    a = a + eps * torch.eye(B, device=a.device, dtype=a.dtype)

    evals, evecs = torch.linalg.eigh(a)
    idx = torch.argsort(evals, descending=True)
    evals = evals[idx]
    evecs = evecs[:, idx]

    return evecs[:, :opt.h_dim], evals

In [6]:
def kPCA_SV(X, Y=None, Z=None, eps=1e-6):
    a = X @ X.T
    B = a.size(0)

    oneN = torch.ones(B, B, device=a.device, dtype=a.dtype) / B
    a = a - oneN @ a - a @ oneN + oneN @ a @ oneN

    a = 0.5 * (a + a.T)  # numerical symmetry
    a = a + eps * torch.eye(B, device=a.device, dtype=a.dtype)

    evals, evecs = torch.linalg.eigh(a)
    idx = torch.argsort(evals, descending=True)
    evals = evals[idx]
    evecs = evecs[:, idx]

    return evecs[:, :opt.h_dim], evals

In [7]:
model_1V = torch.load(f'./out/Final-1V.tar', map_location=opt.device, weights_only=False)
model_2V = torch.load(f'./out/Final.tar', map_location=opt.device, weights_only=False)

In [8]:
# Single View Models
net_im_en_1V = model_1V['net1']
net_im_de_1V = model_1V['net3']

# Dual View Models
net_im_en_2V = model_2V['net1']
net_la_en_2V = model_2V['net2']

net_im_de_2V = model_2V['net3']
net_la_de_2V = model_2V['net4']

In [9]:
net_im_en_1V.eval()
net_im_de_1V.eval()

net_im_en_2V.eval()
net_la_en_2V.eval()
net_im_de_2V.eval()
net_la_de_2V.eval()

Net4(
  (fc1): Linear(in_features=16, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=10, bias=True)
)

In [10]:
all_data_im, all_data_la = [], []

im_en_1V = []
im_en_2V, la_en_2V = [], []

im_de_1V = []
im_de_2V, la_de_2V = [], []

im_loss_1V = []
im_loss_2V, la_loss_2V = [], []

In [11]:
recon_loss1 = torch.nn.MSELoss()
recon_loss2 = torch.nn.BCEWithLogitsLoss()

In [12]:
with torch.no_grad():
    for i, (im, la) in enumerate(xtest):
        if i > i_val * 100:
            break

        im = im.to(opt.device)
        la = la.to(opt.device)

        all_data_im.append(im.cpu())
        all_data_la.append(la.cpu())

        # Single View
        im_en_1V.append(net_im_en_1V(im).cpu())

        # Dual View
        im_en_2V.append(net_im_en_2V(im))
        la_en_2V.append(net_la_en_2V(la))

In [13]:
all_data_im = torch.cat(all_data_im, dim=0)[:1000]
all_data_la = torch.cat(all_data_la)[:1000]

im_en_1V = torch.cat(im_en_1V, dim=0)[:1000]
im_en_2V = torch.cat(im_en_2V, dim=0)[:1000]
la_en_2V = torch.cat(la_en_2V, dim=0)[:1000]

In [14]:
with torch.no_grad():
    for i in range(i_val):
        start, stop = i * 100, (i + 1) * 100

        h_1V, _ = kPCA_SV(im_en_1V[start:stop])
        h_2V, _ = kPCA(im_en_2V[start:stop], la_en_2V[start:stop])

        U_1V = im_en_1V[start:stop].T @ h_1V
        U_2V = im_en_2V[start:stop].T @ h_2V
        V_2V = la_en_2V[start:stop].T @ h_2V

        im_1V_tilde = net_im_de_1V(h_1V @ U_1V.T)
        im_2V_tilde = net_im_de_2V(h_2V @ U_2V.T)
        la_2V_tilde = net_la_de_2V(h_2V @ V_2V.T)

        # Calculate clean losses
        im_true = all_data_im[start:stop].to(opt.device)
        la_true = all_data_la[start:stop].to(opt.device)

        im_loss_1V.append(recon_loss1(im_1V_tilde, im_true).cpu())
        im_loss_2V.append(recon_loss1(im_2V_tilde, im_true).cpu())
        la_loss_2V.append(recon_loss2(la_2V_tilde.view(-1, 10), la_true.view(-1, 10)).cpu())

        if i == 0:
            im_de_1V.append(im_1V_tilde.cpu()[:10])
            im_de_2V.append(im_2V_tilde.cpu()[:10])
            la_de_2V.append(la_2V_tilde.cpu()[:10])

## Attack
### Creating Adversarial Batches

In [15]:
epsilon = 0.2

data_im = all_data_im.cpu()
data_la = all_data_la.cpu()

adv_batches_fgsm, adv_batches_bim = [], []
fgsm_im_tilde, fgsm_la_tilde = [], []
bim_im_tilde, bim_la_tilde = [], []
fgsm_losses_im, bim_losses_im = [], []
fgsm_losses_la, bim_losses_la = [], []

In [16]:
FGSM_1V = WB_Attack(net_in1=net_im_en_1V, net_out1=net_im_de_1V, kPCA=kPCA_SV, opt=opt)
FGSM_2V = WB_Attack(net_in1=net_im_en_2V, net_in2=net_la_en_2V, net_out1=net_im_de_2V, net_out2=net_la_de_2V, kPCA=kPCA, opt=opt)

In [17]:
if model == '1V':
    attack = FGSM_1V
else:
    attack = FGSM_2V

In [18]:
for i in range(i_val):
    print(f'Processing batch {i + 1}/{i_val}...')
    start, stop = i * 100, (i + 1) * 100

    im_batch = data_im[start:stop]
    la_batch = data_la[start:stop] if model == '2V' else None

    adv_batch_fgsm, _ = attack.create_adversarial_batch_fgsm(im_batch, la_batch, epsilon=epsilon)
    adv_batch_bim, _ = attack.create_adversarial_batch_bim(im_batch, la_batch, epsilon=epsilon)

    adv_batches_fgsm.append((adv_batch_fgsm.cpu(), la_batch))
    adv_batches_bim.append((adv_batch_bim.cpu(), la_batch))

    fgsm_x_tilde, fgsm_y_tilde, _ = attack.run_model_batch(adv_batch_fgsm.to(opt.device), la_batch)
    bim_x_tilde, bim_y_tilde, _ = attack.run_model_batch(adv_batch_bim.to(opt.device), la_batch)

    fgsm_im_tilde.append(fgsm_x_tilde.cpu())
    fgsm_la_tilde.append(fgsm_y_tilde)
    bim_im_tilde.append(bim_x_tilde.cpu())
    bim_la_tilde.append(bim_y_tilde)

Processing batch 1/10...
Processing batch 2/10...
Processing batch 3/10...
Processing batch 4/10...
Processing batch 5/10...
Processing batch 6/10...
Processing batch 7/10...
Processing batch 8/10...
Processing batch 9/10...
Processing batch 10/10...


### Adversarial Loss

In [19]:
for i in range(i_val):
    print(f'Evaluating batch {i + 1}/{i_val}...')
    start, stop = i * 100, (i + 1) * 100

    im_batch = data_im[start:stop]
    la_batch = data_la[start:stop]

    adv_batch_fgsm, _ = adv_batches_fgsm[i]
    adv_batch_bim, _ = adv_batches_bim[i]

    fgsm_loss_im = recon_loss1(fgsm_im_tilde[i].to(opt.device), im_batch.to(opt.device)).cpu()
    fgsm_loss_la = recon_loss2(fgsm_la_tilde[i].to(opt.device), la_batch.to(opt.device)).cpu() if model == '2V' else None
    bim_loss_im = recon_loss1(bim_im_tilde[i].to(opt.device), im_batch.to(opt.device)).cpu()
    bim_loss_la = recon_loss2(bim_la_tilde[i].to(opt.device), la_batch.to(opt.device)).cpu() if model == '2V' else None

    fgsm_losses_im.append(fgsm_loss_im.item())
    fgsm_losses_la.append(fgsm_loss_la.item()) if model == '2V' else None
    bim_losses_im.append(bim_loss_im.item())
    bim_losses_la.append(bim_loss_la.item()) if model == '2V' else None

Evaluating batch 1/10...
Evaluating batch 2/10...
Evaluating batch 3/10...
Evaluating batch 4/10...
Evaluating batch 5/10...
Evaluating batch 6/10...
Evaluating batch 7/10...
Evaluating batch 8/10...
Evaluating batch 9/10...
Evaluating batch 10/10...


In [20]:
print(f'---- Evaluation Results for Model----')
print(f'\t\tClean\t\tFGSM\t\tBIM')

print(f'Image Loss:\t{torch.mean(torch.tensor(im_loss_1V)):.4f}\t\t{torch.mean(torch.tensor(fgsm_losses_im)):.4f}\t\t{torch.mean(torch.tensor(bim_losses_im)):.4f}')
print(f'Label Loss:\t{torch.mean(torch.tensor(la_loss_2V)):.4f}\t\t{torch.mean(torch.tensor(fgsm_losses_la)):.4f}\t\t{torch.mean(torch.tensor(bim_losses_la)):.4f}')

---- Evaluation Results for Model----
		Clean		FGSM		BIM
Image Loss:	0.0078		0.0352		0.0089
Label Loss:	1.8656		nan		nan


In [21]:
im_loss_1V = [loss.item() for loss in im_loss_1V]
im_loss_2V = [loss.item() for loss in im_loss_2V]
la_loss_2V = [loss.item() for loss in la_loss_2V]

In [ ]:
# # Save the losses to a CSV file
# results_df = pd.DataFrame({
#     'model': [model] * i_val,

#     'Clean_IM_Loss': im_loss_1V,
#     'FGSM_IM_Loss': fgsm_losses_im,
#     'BIM_IM_Loss': bim_losses_im,

#     'Clean_LA_Loss': im_loss_2V if model == '2V' else [None] * i_val,
#     'FGSM_LA_Loss': fgsm_losses_la if model == '2V' else [None] * i_val,
#     'BIM_LA_Loss': bim_losses_la if model == '2V' else [None] * i_val,
# })

# # Append to file if it exists, otherwise create new
# results_file = 'attack_results.csv'
# if os.path.exists(results_file):
#     results_df.to_csv(results_file, mode='a', header=False, index=False)
# else:
#     results_df.to_csv(results_file, index=False)